In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import torch
from hydra.utils import instantiate
from hydra import initialize, compose
import hydra
import torch.nn as nn

import wandb

from data.dataManager import DataManager
from model.modelCreator import ModelCreator
from omegaconf import OmegaConf
from scripts.run import setup_model, load_model_instance
from utils.HighLevelFeatsAtlasReg import HighLevelFeatures_ATLAS_regular 
from utils.HLF.atlasgeo import AtlasGeometry 
from matplotlib.patches import Wedge
from matplotlib.collections import PatchCollection
from matplotlib.colors import LogNorm, Normalize
from sklearn.calibration import calibration_curve
from scipy.interpolate import UnivariateSpline
from scipy.stats import binned_statistic
import pandas as pd


In [2]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path="config")
cfg=compose(config_name="config_layers.yaml")
wandb.init(tags = [cfg.data.dataset_name], project=cfg.wandb.project, entity=cfg.wandb.entity, config=OmegaConf.to_container(cfg, resolve=True), mode='disabled')

In [3]:
new_model = False
if new_model:
    self = setup_model(cfg)
else:
    config = OmegaConf.load(cfg.config_path)
    config.gpu_list = cfg.gpu_list
    config.load_state = 1
    self = setup_model(config)
    self._model_creator.load_state(config.run_path, self.device, self.optimiser)
    self.data_mgr.apply_stats_and_build_loaders(
        self.model.feature_min.cpu(), 
        self.model.feature_max.cpu()
    )


In [ ]:
model = self.model                                                                                                                                                                      
model.eval()                                                                                                                                                                              
device = self.device

N_BATCHES = 5                                                                                                                                                                             
records = []
                                                                                                                                                                                        
with torch.no_grad():
    for batch_idx, (x, x0, u, E) in enumerate(self.data_mgr.val_loader):
        if batch_idx >= N_BATCHES:                                                                                                                                                        
            break
        x  = x.to(device).float()                                                                                                                                                         
        x0 = x0.to(device).float()                                                                                                                                                        
        u  = u.to(device).to(torch.int32)
        x_frac = self._reduceBCE(x)                                                                                                                                                       
                                                                                                                                                                                        
        output = model((x_frac, x0, u))                                                                                                                                                   
        post_logits, post_samples, output_hits, output_activations = output                                                                                                               
                                                                                                                                                                                        
        beta = getattr(model, '_beta_hits', 1.0)
        ste_mask = model._hit_smoothing(output_hits, beta=beta)                                                                                                                           
        physics_recon = ste_mask * output_activations                                                                                                                                     

        feat_gt    = model.feature_extractor(x_frac)                                                                                                                                      
        feat_recon = model.feature_extractor(physics_recon)
                                                                                                                                                                                        
        raw_lw = model._get_normalized_layer_weights(device, attr="feature_layer_weights")

        B = x_frac.size(0)                                                                                                                                                                
        num_layers = model._config.data.z
        delta = 1.0                                                                                                                                                                       
        alpha = getattr(model._config.model, 'asym_alpha', 2.0)
        asym_centre_only = getattr(model._config.model, 'asym_centre_only', 0)                                                                                                            
        # IQR uses 25th and 75th percentiles (matches compute_physics_loss)
        q = torch.tensor([0.25, 0.75], dtype=torch.float32, device=device)
                                                                                                                                                                                        
        for key in feat_gt.keys():
            if "E_" in key:                                                                                                                                                               
                continue

            val_gt    = feat_gt[key].view(B, -1)    # (B, L)                                                                                                                              
            val_recon = feat_recon[key].view(B, -1)  # (B, L)
                                                                                                                                                                                        
            # IQR scaling (25th/75th percentile across batch, per layer)
            quantiles   = torch.quantile(val_gt, q, dim=0)  # (2, L)
            iqr         = quantiles[1] - quantiles[0]        # (L,)
            layer_spread = torch.clamp(iqr.detach(), min=1e-5)  # (L,)
                                                                                                                                                                                        
            val_gt_scaled    = val_gt    / layer_spread                                                                                                                                   
            val_recon_scaled = val_recon / layer_spread

            abs_scaled_error = torch.abs(val_recon_scaled - val_gt_scaled)
            
            # Fraction of batch in the L1 (linear) regime
            is_linear = (abs_scaled_error > delta).float()
            fraction_linear = is_linear.mean(dim=0).cpu().numpy() # (L,)
            
            # Mean scaled error for context
            mean_scaled_error = abs_scaled_error.mean(dim=0).cpu().numpy() # (L,)
                                                                                                                                                                                        
            base_loss = torch.nn.functional.huber_loss(                                                                                                                                   
                val_recon_scaled, val_gt_scaled, reduction='none', delta=delta
            )  # (B, L)                                                                                                                                                                   
                
            apply_asym = alpha != 1.0 and (not asym_centre_only or "center" in key)                                                                                                       
            if apply_asym:
                underpredict_mask = (torch.abs(val_recon_scaled) < torch.abs(val_gt_scaled)).float()                                                                                      
                w = 1.0 + (alpha - 1.0) * underpredict_mask
                base_loss = base_loss * w                                                                                                                                                 

            # Weighted loss: sum over layers with weights, then mean over batch
            # matches: torch.sum(base_loss * lw, dim=1).mean() in compute_physics_loss
            lw = raw_lw.view(1, -1)  # (1, L)
            weighted_per_sample = torch.sum(base_loss * lw, dim=1)  # (B,)
            weighted_loss_scalar = weighted_per_sample.mean().item()

            per_layer_mean = base_loss.mean(dim=0).cpu().numpy()  # (B,) -> (L,) for diagnostics
            lw_np  = raw_lw.cpu().numpy()
            iqr_np = iqr.cpu().numpy()                                                                                                                                            
                                                                                                                                                                                        
            for l_idx, (raw_val, w_val, iqr_val, frac_lin, mean_err) in enumerate(zip(per_layer_mean, lw_np, iqr_np, fraction_linear, mean_scaled_error)):                                                                                        
                records.append({                                                                                                                                                          
                    "batch":           batch_idx,                                                                                                                                         
                    "feature":         key,
                    "layer":           l_idx,                                                                                                                                             
                    "unweighted_loss": float(raw_val),
                    "weighted_loss":   float(raw_val * w_val),                                                                                                                            
                    "layer_weight":    float(w_val),                                                                                                                                      
                    "layer_scale":     float(iqr_val),
                    "total_feature_loss": weighted_loss_scalar,
                    "fraction_linear": float(frac_lin),
                    "mean_scaled_err": float(mean_err)
                })                                                                                                                                                                        
                
df = pd.DataFrame(records)                                                                                                                                                                
print(df.groupby("feature")[["weighted_loss", "total_feature_loss"]].mean().sort_values("weighted_loss", ascending=False))

In [ ]:
summary_df = df.groupby(["feature", "layer"])[["fraction_linear", "mean_scaled_err", "unweighted_loss"]].mean()
print(summary_df)

In [18]:
feat_totals = df.groupby("feature")["weighted_loss"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
feat_totals.plot.bar(ax=ax)
ax.set_ylabel("Mean weighted loss contribution")
ax.set_title("Physics loss dominance by feature\n(averaged over batches, layer-weighted)")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [19]:
layer_mean = df.groupby(["feature", "layer"])["weighted_loss"].mean().reset_index()
features = layer_mean["feature"].unique()
fig, axes = plt.subplots(1, len(features), figsize=(4 * len(features), 4), sharey=False)
if len(features) == 1:
    axes = [axes]

for ax, feat in zip(axes, features):
    sub = layer_mean[layer_mean["feature"] == feat].sort_values("layer")
    ax.bar(sub["layer"], sub["weighted_loss"])
    ax.set_title(feat)
    ax.set_xlabel("Layer index")
    ax.set_ylabel("Weighted loss")

plt.suptitle("Per-layer loss breakdown by feature", y=1.02)
plt.tight_layout()
plt.show()
scale_df = df.groupby(["feature", "layer"])["layer_scale"].first().unstack("layer")
print("\nLayer-scale (feature_extractor median |val_gt| per layer):")
print(scale_df.to_string())

In [7]:
showers = []
incidence_energies = []

# 1. Load data from the manager
for (x, x0, u, E) in self.data_mgr.train_loader:
    showers.append(x.cpu())
    incidence_energies.append(x0.cpu())

showers = torch.cat(showers, dim=0)
incidence_energies = torch.cat(incidence_energies, dim=0)

# 2. Extract features
self.model.feature_extractor.eval()
with torch.no_grad():
    features = self.model.feature_extractor(showers.to(self.device))

# Eta_center: (N, L) — pick the column that corresponds to layer 1
layer1_col = self.model.geo.relevant_layers.index(1) 
eta_layer1 = features["Eta_center"][:, layer1_col].unsqueeze(1) 

# 3. Numpy + sort by E
y = eta_layer1.squeeze(1).cpu().numpy()
E = (incidence_energies.flatten().cpu().numpy() 
     if isinstance(incidence_energies, torch.Tensor) 
     else np.asarray(incidence_energies).flatten())

order = np.argsort(E)
E_s, y_s = E[order], y[order]

# 1. Fit linear \mu(E) and \sigma^2(E) globally
degree = 1

coeffs_mu = np.polyfit(E_s, y_s, degree)
poly_mu = np.poly1d(coeffs_mu)
mu_local = poly_mu(E_s)

# Calculate residuals and fit variance
R_poly = (y_s - mu_local) ** 2
coeffs_var = np.polyfit(E_s, R_poly, degree)
poly_var = np.poly1d(coeffs_var)

# Prevent negative variance and division by zero
var_local = np.clip(poly_var(E_s), 1e-6, None)
sig_local = np.sqrt(var_local)

# 2. Compute absolute Z-scores
z_scores = np.abs(y_s - mu_local) / sig_local

# 3. Calculate Empirical Survival Probabilities (CCDF)
sorted_z = np.sort(z_scores)
# p_greater_than_z represents P(Z > z)
p_greater_than_z = 1.0 - np.arange(1, len(sorted_z) + 1) / len(sorted_z)

# Map each sample's Z-score to its survival probability
indices = np.searchsorted(sorted_z, z_scores)
indices = np.clip(indices, 0, len(sorted_z) - 1)
sample_probs = p_greater_than_z[indices]

# 4. Apply Power-Law Scaled Inverse Weighting with Hard Cap
gamma = 0.5
w_max = 500.0

raw_weights = sample_probs ** (-gamma)
sample_weights = np.clip(raw_weights, 1.0, w_max)

# Mean-normalize weights so the total effective batch size remains constant
normalized_weights = sample_weights / sample_weights.mean()

# ── PLOTTING THE DISTRIBUTIONS ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
bins = 100

# Plot 1: Original Feature Distribution
ax1.hist(y_s, bins=bins, color='steelblue', alpha=0.8, log=True, density=True)
ax1.set_xlabel(r'Feature Value ($\eta$ Layer 1)')
ax1.set_ylabel('Raw Counts')
ax1.set_title('Original Feature Distribution')
ax1.grid(True, alpha=0.3, ls='--')

# Plot 2: Implied Weighted Feature Distribution
# We pass the normalized_weights to the weights argument of plt.hist
ax2.hist(y_s, bins=bins, weights=normalized_weights, color='darkorange', alpha=0.8, log=True, density=True)
ax2.set_xlabel(r'Feature Value ($\eta$ Layer 1)')
ax2.set_ylabel('Effective Counts (Importance Weighted)')
ax2.set_title(rf'Implied Distribution ($\gamma={gamma}$, $W_{{max}}={w_max}$)')
ax2.grid(True, alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Assuming you already have: E_s, y_s, E_fine, mu_fine (median), sig_fine (scaled MAD)
# from the robust binned spline fitting we discussed earlier.

# 1. Calculate the continuous robust Z-score for every data point
# Evaluate the spline at the exact energy of each data point to get its local mu and sigma
mu_local = spl_mu(E_s)
sig_local = spl_var(E_s)

# Prevent division by zero just in case
sig_local = np.clip(sig_local, 1e-4, None) 

# Absolute Robust Z-score: |y - median| / MAD_sigma
z_scores = np.abs(y_s - mu_local) / sig_local

# 2. Plotting Setup
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ─── PLOT 1: THE SURVIVAL FUNCTION (CCDF) ─────────────────────────────────────
# This shows exactly what fraction of your data exceeds a given Z-score.
# It is the mathematically correct way to view a 10-sigma tail.

sorted_z = np.sort(z_scores)
# Calculate 1 - CDF
p_greater_than_z = 1.0 - np.arange(1, len(sorted_z) + 1) / len(sorted_z)

ax1.plot(sorted_z, p_greater_than_z, color='indigo', lw=2.5)
ax1.set_yscale('log')
ax1.set_xlim(0, max(z_scores) * 1.05)

# Floor the y-axis at slightly below your lowest density (e.g., 1e-6)
ax1.set_ylim(1 / len(z_scores), 1.0) 

ax1.set_xlabel("Absolute Z-Score threshold ($z$)")
ax1.set_ylabel("Fraction of Data > $z$")
ax1.set_title("Z-Score Survival Function (CCDF)")
ax1.grid(True, which="both", ls="--", alpha=0.4)

# Add reference lines for standard sigma thresholds
for sig in [1, 3, 5, 10]:
    if sig < max(z_scores):
        ax1.axvline(sig, color='gray', ls=':', alpha=0.7)
        ax1.text(sig + 0.1, 0.5, f"{sig}$\sigma$", rotation=90, color='gray', va='center')


# ─── PLOT 2: CORE HEXBIN + TAIL SCATTER ───────────────────────────────────────
# Separating the dense core from the sparse tail prevents the tail from vanishing 
# into empty histogram bins.

mask_core = z_scores < 3.0
mask_tail = z_scores >= 3.0

# Plot the dense core using hexbin (often looks cleaner than hist2d for physics data)
hb = ax2.hexbin(E_s[mask_core], z_scores[mask_core], gridsize=60, 
                cmap='Blues', bins='log', mincnt=1)

# Overlay the sparse tail as individual points
ax2.scatter(E_s[mask_tail], z_scores[mask_tail], 
            color='crimson', s=4, alpha=0.4, label='Tail ($Z \geq 3$)')

ax2.set_xlabel("Incident energy $E$")
ax2.set_ylabel("Absolute Z-Score")
ax2.set_title("Energy vs Z-Score (Core + Scatter Overlay)")
ax2.legend(loc='upper right')
fig.colorbar(hb, ax=ax2, label='Log Counts (Core only)')

plt.tight_layout()
plt.show()

In [ ]:
self.evaluate_ae(self.data_mgr.val_loader ,epoch=158)

In [ ]:
post_samples = self.post_samples[:, self._config.model.cond_p_size:]
# plot magnetization of post_samples

mean_magnetization = post_samples.mean(dim=0).cpu().numpy()
plt.figure(figsize=(10,6))
plt.hist(mean_magnetization, bins=np.arange(0,1.05,0.05), alpha=0.7, color='blue', edgecolor='black')
plt.xlabel('Visible Unit Index')
plt.ylabel('Mean Magnetization')
plt.title('Mean Magnetization of Non Conditioning Latent Nodes from Posterior')

In [ ]:
self.evaluate_ae(self.data_mgr.train_loader ,epoch=158)

In [ ]:
print(f"Maximum incidence energy: {self.incident_energy.max().item()} MeV, Minimum incidence energy: {self.incident_energy.min().item()} MeV")

In [ ]:
encoded_energies = self.post_samples[:, :self._config.model.cond_p_size]
print(encoded_energies.shape)
means = encoded_energies.mean(axis=0)
num_nodes = len(means)

# Define grouping parameters: (start_idx, end_idx, label, color, hatch)
e_inc_bits = self._config.model.lin_bits + self._config.model.sqrt_bits + self._config.model.log_bits
groups = [
    (0, self._config.model.lin_bits, 'Linear', '#3498db', '/'),      # Blue, forward slash
    (self._config.model.lin_bits, self._config.model.lin_bits + self._config.model.sqrt_bits, 'Square Root', '#e74c3c', '\\'), # Red, back slash
    (self._config.model.lin_bits + self._config.model.sqrt_bits, e_inc_bits, 'Log', '#2ecc71', 'x'),         # Green, cross pattern
    (e_inc_bits, e_inc_bits+self._config.model.u_bits, 'U1', '#95a5a6', 'o'),         # Grey, circle pattern
    (e_inc_bits+self._config.model.u_bits, e_inc_bits+self._config.model.u_bits*2, 'U2', '#f39c12', 'O'),         # Orange, large circle pattern
    (e_inc_bits+self._config.model.u_bits*2, e_inc_bits+self._config.model.u_bits*3, 'U3', '#9b59b6', '*') ,        # Purple, star pattern
    (e_inc_bits+self._config.model.u_bits*3, e_inc_bits+self._config.model.u_bits*4, 'U4', '#34495e', '+'),         # Dark Blue, plus pattern
    (e_inc_bits+self._config.model.u_bits*4, e_inc_bits+self._config.model.u_bits*5, 'U5', '#16a085', 'D'),         # Teal, diamond pattern
]

plt.figure(figsize=(12, 6))
indices = np.arange(num_nodes)

# Plot each group separately to apply different styles and labels for the legend
for start, end, label, color, hatch in groups:
    plt.bar(
        indices[start:end], 
        means[start:end], 
        color=color, 
        hatch=hatch, 
        label=label, 
        edgecolor='white', 
        alpha=0.85
    )

# Aesthetic improvements
plt.xlabel('Node Index', fontsize=12, fontweight='bold')
plt.ylabel('Average Magnetization', fontsize=12, fontweight='bold')
plt.title('Average Magnetization per Encoding Group', fontsize=14, pad=15, fontweight='bold')

# Ensure all node indices are shown on the x-axis
plt.xticks(indices, rotation=90, fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.legend(
    title='Node Grouping', 
    frameon=True, 
    shadow=True,
    loc='lower left', 
    bbox_to_anchor=(0.75, 1.02), # Pushes legend just above the top edge
    ncol=3 # Splits the 8 items cleanly into 2 horizontal rows
)
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# collect u vectors from dataloader, concat and plot histograms for each of u_i
u_list = []
for (x, x0, u, E) in self.data_mgr.val_loader:
    u_list.append(u.cpu().numpy())

u = np.concatenate(u_list, axis=0)
print(u.shape)

plt.figure(figsize=(12, 6))
for i in range(5):
    plt.subplot(2, 3, i+1)
    plt.hist(u[:, i]*7, bins=30, alpha=0.7, color='blue', edgecolor='black')
    plt.title(rf'$U_{i+1}$ Distribution')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.xlim(0, 8)
    plt.yscale('log')
plt.tight_layout()

In [ ]:
# 1. Isolate the specific kernel/stride geometry from your FirstSubdecoderLayers
# in_channels=1, out_channels=1 is enough to see the spatial math
offending_layer = nn.ConvTranspose3d(
    in_channels=1, 
    out_channels=1, 
    kernel_size=(3, 5, 5), 
    stride=(1, 1, 2), 
    padding=(1, 0, 0)
)

# 2. Force weights to 1.0 and biases to 0.0
# This turns the layer into a pure counter of "how many times did the kernel overlap here?"
nn.init.constant_(offending_layer.weight, 1.0)
nn.init.constant_(offending_layer.bias, 0.0)

# 3. Create a flat "blank canvas" input tensor of 1s
dummy_input = torch.ones(1, 1, 5, 7, 7)

# 4. Push the uniform signal through the untrained layer
with torch.no_grad():
    output = offending_layer(dummy_input)

# 5. Extract a 1D slice along the r-dimension (the 3rd spatial dim)
# We take the center of the z and phi dimensions to avoid edge padding effects
z_center, phi_center = output.shape[2] // 2, output.shape[3] // 2
r_profile = output[0, 0, z_center, phi_center, :].numpy()

# 6. Plot the result
plt.figure(figsize=(10, 4))
plt.plot(r_profile, marker='o')
plt.title("Pure Structural Overlap in r-dimension (Weights = 1)")
plt.xlabel("r index")
plt.ylabel("Signal Amplitude (Kernel Overlaps)")
plt.grid(True)
plt.show()

In [ ]:
print(self.model.encoder.gray_codec.tables.keys())
print(hasattr(self._config, 'use_gray_code') and self._config.use_gray_code)

In [ ]:
def gray_code(n):
    return n ^ (n >> 1)

def hamming_dist(a, b):
    return bin(a ^ b).count('1')

def check_19_bits():
    # Resolution 19 bits (0 to 2^19 - 1)
    # Encoded in 20 bits
    n_data = 19
    window_size = 2**n_data
    
    # Check Offset 1
    offset = 1
    start_val = gray_code(offset)
    end_val = gray_code(offset + window_size - 1)
    
    print(f"Data Bits: {n_data}")
    print(f"Window Size: {window_size}")
    print(f"Offset: {offset}")
    print(f"Start Index: {offset} -> Code: {bin(start_val)}")
    print(f"End Index: {offset + window_size - 1} -> Code: {bin(end_val)}")
    print(f"Hamming Distance: {hamming_dist(start_val, end_val)}")

check_19_bits()


In [ ]:
# self.evaluate_ae(self.data_mgr.train_loader ,epoch=0)
# self.generate_plots(epoch=0, key="ae")
# self.evaluate_ae(self.data_mgr.train_loader ,epoch=0)
# self.generate_plots(epoch=0, key="ae")
self.fit_ae(158)
self.evaluate_ae(self.data_mgr.train_loader ,epoch=158)
self.generate_plots(epoch=158, key="ae")

In [ ]:
lr = self.optimiser.param_groups[0]["lr"]
print(f"Learning Rate: {lr}")
# checking the internal state buffer
print("Optimizer state len:", len(self.optimiser.state))
print("Optimizer state keys:", self.optimiser.state.keys())

# checking the iteration step
# Note: 'state' is a dict mapping params to their state. 
# We just need to check the first parameter found to see its step count.
if len(self.optimiser.state) > 0:
    # first_param_key = list(self.optimiser.state.keys())[0]
    print("Current Step Count:", self.optimiser.state[first_param_key]['step'])
else:
    print("Optimizer state is empty (Fresh).")

In [ ]:
# Access the specific layer
final_layer = self.model.decoder.subdecoders[3]._subdecoder_layers[3]

print(f"Layer type: {type(final_layer)}")
print(f"Bias parameter present: {final_layer.bias is not None}")
if final_layer.bias is not None:
    print(f"Bias requires grad: {final_layer.bias.requires_grad}")
    print(f"Bias data (first 5): {final_layer.bias.data.flatten()[:5]}")

In [ ]:
# 1. Connect to the specific run
api = wandb.Api()
run_path = "caloqvae/caloqvae/1rdxku2w"  # Replace this!
run = api.run(run_path)

# 2. Define the key for the bias gradient
# This must match exactly what you saw in the UI
key = "gradients/decoder.subdecoders.3._subdecoder_layers.3.bias"

# 3. Download the history
print("Downloading run history...")
history = run.history(keys=[key])

reconstructed_means = []
steps = []

# 4. Iterate through rows to reconstruct the mean
for index, row in history.iterrows():
    hist_data = row[key]
    
    # Check if data exists for this step
    if isinstance(hist_data, dict) and 'bins' in hist_data and 'values' in hist_data:
        bins = np.array(hist_data['bins'])
        counts = np.array(hist_data['values'])
        
        # Calculate bin centers
        # WandB bins are edges, so centers are average of adjacent edges
        bin_centers = (bins[:-1] + bins[1:]) / 2
        
        # Calculate Weighted Average (Approximate Mean)
        if np.sum(counts) > 0:
            approx_mean = np.average(bin_centers, weights=counts)
            reconstructed_means.append(approx_mean)
            steps.append(index)

# 5. Plot the result
plt.figure(figsize=(10, 6))
plt.plot(steps, reconstructed_means, label='Reconstructed Bias Gradient Mean', color='red', alpha=0.7)
plt.xlabel("Step")
plt.ylabel("Gradient Magnitude")
plt.title("Reconstructed Bias Gradients (From Histogram Bins)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
import h5py
from collections import defaultdict


def test_custom_leakage(cfg):
    print(f"--- Loading Custom Dataset: {cfg.data.path} ---")
    
    # 1. Load Data
    with h5py.File(cfg.data.path, 'r') as f:
        # Load all to memory for the test (or slice if file is huge)
        showers = torch.tensor(f["showers"][:]).float()
        energies = torch.tensor(f["incident_energies"][:]).float().squeeze()

    energies_np = energies.numpy()
    
    # 2. Replicate Your EXACT "Custom" Binning Logic
    print("Applying Linear Binning...")
    min_e = energies_np.min()
    max_e = energies_np.max()
    num_bins = 15
    energy_bin_edges = np.linspace(min_e, max_e + 1e-6, num_bins + 1)
    bin_ids = np.digitize(energies_np, energy_bin_edges, right=False)

    # 3. Build Indices (Stratified)
    bin_to_indices = defaultdict(list)
    # Note: Insertion order depends on when a bin is first encountered in the data
    for i, b in enumerate(bin_ids):
        bin_to_indices[b].append(i)

    train_idx, val_idx, test_idx = [], [], []
    
    # To track which bins are losing data
    bin_stats = {} 

    for b_id, indices in bin_to_indices.items():
        n = len(indices)
        n_train = int(cfg.data.frac_train_dataset * n)
        n_val = int(cfg.data.frac_val_dataset * n)
        
        bin_stats[b_id] = {'n': n, 'n_train': n_train, 'n_val': n_val}

        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train:n_train + n_val])
        test_idx.extend(indices[n_train + n_val:])

    # 4. Compare Stratified vs Global
    real_train_len = len(train_idx)
    
    total_len = len(showers)
    global_train_slice = int(np.floor(total_len * cfg.data.frac_train_dataset))
    
    diff = global_train_slice - real_train_len
    
    print("\n" + "="*40)
    print("      CUSTOM DATASET DIAGNOSTICS      ")
    print("="*40)
    print(f"Total Events:      {total_len}")
    print(f"Stratified Length: {real_train_len}")
    print(f"Global Slice:      {global_train_slice}")
    print("-" * 20)
    
    if diff > 0:
        print(f"[FAIL] LEAK DETECTED: {diff} events")
        print(f"The Global Slicer is grabbing {diff} extra events.")
        
        # Forensic: Which bin did we steal from?
        # The leak eats into the START of the concatenated val_idx list.
        # The start of val_idx corresponds to the FIRST bin key in bin_to_indices.
        
        first_bin_key = list(bin_to_indices.keys())[0]
        leaked_indices = val_idx[:diff]
        leaked_energies = energies_np[leaked_indices]
        
        print(f"\nFATAL ERROR: The leaked events belong to Bin ID {first_bin_key}")
        print(f"This bin was the FIRST bin encountered in your dataset.")
        print(f"Leaked Energy Values: {leaked_energies}")
        print(f"\nIMPACT: Your validation set has NO events from Bin {first_bin_key} for the first {diff} slots.")
        print("This explains the 'noisy' chi2: You are validating on a population that is missing specific energies.")
    else:
        print("[PASS] No leakage detected.")

test_custom_leakage(config)

In [ ]:
self.model.eval()
ar_input_size = self._config.data.z * self._config.data.r * self._config.data.phi
bs = [batch[0].shape[0] for batch in self.data_mgr.val_loader]
output_hits = torch.zeros(sum(bs), ar_input_size)
with torch.no_grad():
    for i, (x, x0) in enumerate(self.data_mgr.val_loader):
        idx1, idx2 = int(np.sum(bs[:i])), int(np.sum(bs[:i+1]))
        x = x.to(self.device)
        x0 = x0.to(self.device)

        x_reduce = self._reduce(x, x0)
        # Forward pass
        output = self.model((x_reduce, x0))

        output_hits_temp = output[4]
        output_hits[idx1:idx2, :] = output_hits_temp.cpu()


In [ ]:
def plot_logit_distribution(hits_logits, title="Logit Distribution"):
    """
    hits_logits: Tensor of shape (Batch, Flat_Voxels)
    """
    # Flatten everything to see the global distribution
    flat_logits = hits_logits.detach().cpu().flatten().numpy()
    
    plt.figure(figsize=(10, 5))
    plt.hist(flat_logits, bins=100, log=True, color='skyblue', edgecolor='black')
    plt.title(title)
    plt.xlabel("Logit Value (Sigmoid input)")
    plt.ylabel("Count (Log Scale)")
    plt.axvline(x=0, color='r', linestyle='--') # Decision boundary
    plt.grid(True, which="both", ls="-", alpha=0.5)
    plt.show()

plot_logit_distribution(output_hits, title="Logit Distribution of Reconstructed Hits")

In [ ]:
def visualize_spatial_hits(hits_logits, z_dim, r_dim, phi_dim):
    """
    Reshapes logits to detector geometry and plots projections.
    """
    # 1. Sigmoid to get probabilities [0, 1]
    probs = torch.sigmoid(hits_logits).detach().cpu()
    
    # 2. Reshape: (Batch, Z, R, Phi)
    # Adjust the order (Z, R, Phi) if your config flattens differently!
    probs_3d = probs.view(-1, z_dim, phi_dim, r_dim)
    
    # 3. Average over the batch to see "what the model usually thinks is a hit"
    avg_hits = probs_3d.mean(dim=0) # Shape: (Z, Phi, R)
    
    # 4. Project (Sum) over one dimension to visualize 2D heatmaps
    # Example: Project over Phi to see Z-R view
    zr_projection = avg_hits.sum(dim=2) 
    
    plt.figure(figsize=(12, 5))
    plt.imshow(zr_projection, origin='lower', aspect='auto', cmap='magma')
    plt.colorbar(label='Summed Hit Probability')
    plt.title("Average Hit Probability (Z-R Projection)")
    plt.xlabel("R bins")
    plt.ylabel("Z bins")
    plt.show()

visualize_spatial_hits(output_hits, z_dim=cfg.data.z, r_dim=cfg.data.r, phi_dim=cfg.data.phi)

In [ ]:
def analyze_activation_uncertainty(engine, single_batch_x, single_batch_x0, n_samples=20):
    engine.model.eval()
    # We only care about the hits output (index 4)
    sampled_hits = []
    
    with torch.no_grad():
        x_reduce = engine._reduce(single_batch_x, single_batch_x0)
        
        for _ in range(n_samples):
            # The model adds new random noise each forward pass
            _, _, _, _, output_hits = engine.model((x_reduce, single_batch_x0))
            
            # This is your binary output (0 or 1)
            # Make sure your model returns the heaviside output, not raw logits
            # If your forward returns logits, apply your GumbelMod manually here
            sampled_hits.append(output_hits)
            
    # Stack: (N_samples, Batch, Z*R*Phi)
    stack = torch.stack(sampled_hits)
    
    # Calculate Mean (Probability) and Std (Uncertainty)
    mean_map = stack.mean(dim=0)
    uncertainty_map = stack.std(dim=0)
    
    return mean_map, uncertainty_map

mean_map, uncertainty_map = analyze_activation_uncertainty(self, x, x0, n_samples=10)

In [ ]:
def plot_layer_on_ax(ax, hlf_instance, layer_id, energy_data, title=None, 
                     norm=None, cmap='rainbow'):
    """
    Draws a single calorimeter layer directly onto a provided Matplotlib axis.
    """
    # 1. Inject state into HLF
    hlf_instance.single_event_energy = energy_data
    hlf_instance.current_layer = str(layer_id)
    
    # 2. Get Geometry
    r0, r1, a0, a1, e = hlf_instance.get_sector_arrays(hlf_instance.current_layer)

    # 3. Precompute Transform (Equal Bin Area)
    transform = hlf_instance._make_equal_bin_transform(r0, r1)
    r0p, r1p = transform(r0), transform(r1)

    # 4. Build Wedges
    patches = []
    for inner, outer, start, end in zip(r0p, r1p, a0, a1):
        width = outer - inner
        patches.append(Wedge((0, 0), outer, start, end, width=width))

    # 5. Create Collection
    # Use the passed norm and cmap
    pc = PatchCollection(patches, cmap=cmap, norm=norm, edgecolor="grey", linewidths=0.1)
    pc.set_array(e)
    ax.add_collection(pc)
    
    # 6. Styling
    Rmax = r1p.max()
    ax.set_xlim(-Rmax - 0.1, Rmax + 0.1)
    ax.set_ylim(-Rmax - 0.1, Rmax + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=10, pad=8)
    
    return pc

def visualize_hit_uncertainty(mean_map, uncertainty_map, binning_path, 
                              layers_to_plot=[0, 1, 2, 3, 4], 
                              device="cpu"):
    """
    Plots the Mean (Probability) and Uncertainty (Std Dev) for the specified calorimeter layers.
    
    Args:
        mean_map (Tensor or array): Shape (Flat_Voxels,). Values [0, 1].
        uncertainty_map (Tensor or array): Shape (Flat_Voxels,). Values [0, 0.5].
        binning_path (str): Path to the binning XML/YAML file for geometry.
        layers_to_plot (list): List of integer layer IDs to visualize.
    """
    
    # 1. Setup Geometry and HLF
    # We re-instantiate these to ensure we have the correct transforms
    geo = AtlasGeometry(filename=binning_path)
    HLF = HighLevelFeatures_ATLAS_regular(
        particle='electron', filename=binning_path, relevantLayers=geo.relevant_layers
    )
    
    # 2. Data Prep (Ensure CPU Numpy)
    if isinstance(mean_map, torch.Tensor):
        mean_map = mean_map.detach().cpu().numpy()
    if isinstance(uncertainty_map, torch.Tensor):
        uncertainty_map = uncertainty_map.detach().cpu().numpy()
        
    # Constants based on your previous snippet
    vox_per_layer = 14 * 24 
    
    # 3. Setup Plot Grid
    n_layers = len(layers_to_plot)
    fig, axes = plt.subplots(n_layers, 2, figsize=(10, 4.5 * n_layers), dpi=120)
    
    # Handle single layer case to maintain 2D array structure
    if n_layers == 1:
        axes = np.array([axes])
        
    print(f"Plotting Hit Diagnostics for Layers: {layers_to_plot}")

    # 4. Loop through layers
    for i, layer_id in enumerate(layers_to_plot):
        if layer_id not in geo.relevant_layers:
            print(f"Skipping Layer {layer_id} (not in relevant_layers)")
            continue
            
        # -- Slicing Logic --
        # Map physical layer ID to the flattened index
        layer_idx = geo.relevant_layers.index(layer_id)
        start_idx = layer_idx * vox_per_layer
        end_idx = (layer_idx + 1) * vox_per_layer
        
        # Extract data for this layer
        data_mean = mean_map[start_idx:end_idx]
        data_unc = uncertainty_map[start_idx:end_idx]
        
        # -- Plot Left: Mean (Probability) --
        ax_mean = axes[i, 0]
        pc_mean = plot_layer_on_ax(
            ax_mean, HLF, layer_id, data_mean,
            title=f"Layer {layer_id} | Mean Probability",
            norm=Normalize(vmin=0.0, vmax=1.0),
            cmap='viridis'
        )
        cbar_m = plt.colorbar(pc_mean, ax=ax_mean, fraction=0.046, pad=0.04)
        cbar_m.set_label("P(Hit)", fontsize=9)

        # -- Plot Right: Uncertainty (Bernoulli Std) --
        ax_unc = axes[i, 1]
        pc_unc = plot_layer_on_ax(
            ax_unc, HLF, layer_id, data_unc,
            title=f"Layer {layer_id} | Uncertainty (Std Dev)",
            # Max Std for Bernoulli is 0.5 (at p=0.5)
            norm=Normalize(vmin=0.0, vmax=0.5), 
            cmap='inferno'
        )
        cbar_u = plt.colorbar(pc_unc, ax=ax_unc, fraction=0.046, pad=0.04)
        cbar_u.set_label("Std Dev", fontsize=9)

    plt.tight_layout()
    plt.show()

binning_path = self._config.data.binning_path
# mean_map_flat = mean_map.mean(dim=0) 
# uncertainty_map_flat = uncertainty_map.mean(dim=0)
# visualize_hit_uncertainty(mean_map_flat, uncertainty_map_flat, binning_path, layers_to_plot=[0,1,2,3,12], device=self.device)

In [ ]:
def analyze_activation_comparison(engine):
    """
    Extracts Model Probabilities and Ground Truth Hits to compare their 
    means and variabilities (uncertainties) across the dataset.
    """
    engine.model.eval()

    # 1. Setup storage
    bs = [batch[0].shape[0] for batch in engine.data_mgr.val_loader]
    total_samples = sum(bs)
    ar_input_size = engine._config.data.z * engine._config.data.r * engine._config.data.phi
    
    # Store data on CPU to avoid OOM
    model_probs_all = torch.zeros(total_samples, ar_input_size)
    gt_hits_all = torch.zeros(total_samples, ar_input_size)
    
    current_idx = 0
    
    with torch.no_grad():
        for i, (x, x0) in enumerate(engine.data_mgr.val_loader):
            batch_size = x.shape[0]
            x = x.to(engine.device)
            x0 = x0.to(engine.device)
            
            # Prepare input
            x_reduce = engine._reduce(x, x0)
            
            # 2. Forward pass
            # We need the LOGITS to calculate precise probabilities
            # Assuming outputs[4] contains the raw logits before Gumbel/Heaviside
            outputs = engine.model((x_reduce, x0))
            logits = outputs[4] 
            
            # 3. Model Calculations
            # We use Sigmoid(logits) to get the 'true' probability distribution 
            # predicted by the model, ignoring the Gumbel noise added during eval step.
            probs = torch.sigmoid(logits)
            
            # 4. Ground Truth Calculations
            # Hits are simply where energy > 0
            gt_hits_batch = (x > 0).float()
            
            # 5. Store
            end_idx = current_idx + batch_size
            model_probs_all[current_idx:end_idx] = probs.cpu()
            gt_hits_all[current_idx:end_idx] = gt_hits_batch.cpu()
            
            current_idx = end_idx

    return model_probs_all, gt_hits_all

def visualize_comparison(gt_hits_all, model_probs_all, binning_path, 
                         layers_to_plot=[0, 1, 2, 3, 12], 
                         device="cpu"):
    """
    Plots a 4-column comparison: 
    GT Mean | GT Std | Model Mean | Model Std
    """
    
    # 1. Compute Statistics over the dataset
    
    # --- Ground Truth ---
    # Mean: Average Hit Rate
    gt_mean = gt_hits_all.mean(dim=0)
    # Uncertainty: Standard Deviation of the hits (Variability)
    # This captures the "nuance": Voxels that toggle 0/1 often have high std.
    gt_uncertainty = gt_hits_all.std(dim=0) 
    
    # --- Model ---
    # Mean: Average Probability
    model_mean = model_probs_all.mean(dim=0)
    # Uncertainty: Standard Deviation of the Probabilities
    # If model is Confused (always 0.5) -> Std = 0
    # If model is Variable (toggles 1.0/0.0) -> Std = 0.5
    model_uncertainty = model_probs_all.std(dim=0)

    # 2. Setup Geometry
    geo = AtlasGeometry(filename=binning_path)
    HLF = HighLevelFeatures_ATLAS_regular(
        particle='electron', filename=binning_path, relevantLayers=geo.relevant_layers
    )
    
    # Convert to Numpy
    gt_mean = gt_mean.numpy()
    gt_uncertainty = gt_uncertainty.numpy()
    model_mean = model_mean.numpy()
    model_uncertainty = model_uncertainty.numpy()
    
    vox_per_layer = 14 * 24 
    n_layers = len(layers_to_plot)
    
    # 3. Setup Plot Grid
    fig, axes = plt.subplots(n_layers, 4, figsize=(20, 4.5 * n_layers), dpi=120)
    if n_layers == 1: axes = np.array([axes])
        
    print(f"Plotting Comparison for Layers: {layers_to_plot}")

    for i, layer_id in enumerate(layers_to_plot):
        if layer_id not in geo.relevant_layers:
            continue
            
        layer_idx = geo.relevant_layers.index(layer_id)
        start = layer_idx * vox_per_layer
        end = (layer_idx + 1) * vox_per_layer
        
        # --- Column 1: GT Mean ---
        ax = axes[i, 0]
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, gt_mean[start:end],
            title=f"L{layer_id} | GT Mean Hit Rate",
            norm=Normalize(vmin=0.0, vmax=1.0), cmap='viridis'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)

        # --- Column 2: GT Uncertainty (Variability) ---
        ax = axes[i, 1]
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, gt_uncertainty[start:end],
            title=f"L{layer_id} | GT Variability (Std)",
            norm=Normalize(vmin=0.0, vmax=0.5), cmap='inferno'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)
        
        # --- Column 3: Model Mean ---
        ax = axes[i, 2]
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, model_mean[start:end],
            title=f"L{layer_id} | Model Mean Prob",
            norm=Normalize(vmin=0.0, vmax=1.0), cmap='viridis'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)

        # --- Column 4: Model Uncertainty (Variability) ---
        ax = axes[i, 3]
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, model_uncertainty[start:end],
            title=f"L{layer_id} | Model Variability (Std)",
            norm=Normalize(vmin=0.0, vmax=0.5), cmap='inferno'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

In [ ]:
def visualize_comparison_log(gt_hits_all, model_probs_all, binning_path, 
                             layers_to_plot=[0, 1, 2, 3, 12], 
                             device="cpu"):
    
    # ... [Same Statistics Computation as before] ...
    
    # 1. Compute Statistics
    gt_mean = gt_hits_all.mean(dim=0).numpy()
    gt_uncertainty = gt_hits_all.std(dim=0).numpy()
    model_mean = model_probs_all.mean(dim=0).numpy()
    model_uncertainty = model_probs_all.std(dim=0).numpy()
    
    # ... [Same Geometry Setup] ...
    geo = AtlasGeometry(filename=binning_path)
    HLF = HighLevelFeatures_ATLAS_regular(
        particle='electron', filename=binning_path, relevantLayers=geo.relevant_layers
    )
    
    vox_per_layer = 14 * 24 
    n_layers = len(layers_to_plot)
    
    fig, axes = plt.subplots(n_layers, 4, figsize=(20, 4.5 * n_layers), dpi=120)
    if n_layers == 1: axes = np.array([axes])
        
    print(f"Plotting Log-Scale Comparison for Layers: {layers_to_plot}")

    # DEFINE YOUR FLOOR: 
    # If you have 10,000 events, the rarest possible non-zero frequency is 1e-4.
    # Setting vmin=1e-3 means "Show me anything that happens >0.1% of the time."
    LOG_FLOOR = 1e-3 

    for i, layer_id in enumerate(layers_to_plot):
        if layer_id not in geo.relevant_layers:
            continue
            
        layer_idx = geo.relevant_layers.index(layer_id)
        start = layer_idx * vox_per_layer
        end = (layer_idx + 1) * vox_per_layer
        
        # --- Column 1: GT Mean (LOG SCALE) ---
        ax = axes[i, 0]
        # We add a tiny epsilon to data in case your plotting function 
        # doesn't handle masked values automatically, though LogNorm usually does.
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, gt_mean[start:end],
            title=f"L{layer_id} | GT Mean Hit Rate (Log)",
            # Use LogNorm here
            norm=LogNorm(vmin=LOG_FLOOR, vmax=1.0), 
            cmap='viridis'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)

        # --- Column 2: GT Variability (Linear) ---
        # Keeping this linear is usually better for Std, but you can change it if needed.
        ax = axes[i, 1]
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, gt_uncertainty[start:end],
            title=f"L{layer_id} | GT Variability (Std)",
            norm=Normalize(vmin=0.0, vmax=0.5), cmap='inferno'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)
        
        # --- Column 3: Model Mean (LOG SCALE) ---
        ax = axes[i, 2]
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, model_mean[start:end],
            title=f"L{layer_id} | Model Mean Prob (Log)",
            # Use LogNorm here
            norm=LogNorm(vmin=LOG_FLOOR, vmax=1.0), 
            cmap='viridis'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)

        # --- Column 4: Model Variability (Linear) ---
        ax = axes[i, 3]
        pc = plot_layer_on_ax(
            ax, HLF, layer_id, model_uncertainty[start:end],
            title=f"L{layer_id} | Model Variability (Std)",
            norm=Normalize(vmin=0.0, vmax=0.5), cmap='inferno'
        )
        plt.colorbar(pc, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

In [ ]:
model_probs, gt_hits = analyze_activation_comparison(self)

# 2. Visualize
binning_path = self._config.data.binning_path
visualize_comparison(
    gt_hits, 
    model_probs, 
    binning_path, 
    layers_to_plot=[0,1,2,3,12], 
    device=self.device
)

In [ ]:
visualize_comparison_log(
    gt_hits, 
    model_probs, 
    binning_path, 
    layers_to_plot=[0,1,2,3,12], 
    device=self.device
)

In [ ]:
def plot_calibration_and_sharpness(gt_hits_all, model_probs_all, n_bins=20):
    """
    Plots the Reliability Diagram (Calibration) and the Sharpness Histogram.
    
    Args:
        gt_hits_all: Tensor or array of Ground Truth (0 or 1)
        model_probs_all: Tensor or array of Model Probabilities (0 to 1)
    """
    # 1. Flatten the arrays (We treat all voxels from all events as one giant pool)
    # Note: If memory is an issue, sample a subset (e.g., 1 million points)
    y_true = gt_hits_all.flatten().numpy()
    y_prob = model_probs_all.flatten().numpy()
    
    print(f"Analyzing {len(y_true)} predictions...")

    # 2. Compute Calibration Curve
    # 'prob_true' is the fraction of positives in each bin
    # 'prob_pred' is the mean predicted probability in each bin
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy='uniform')

    # 3. Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # --- Plot A: Calibration Curve (Reliability) ---
    ax1.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')
    ax1.plot(prob_pred, prob_true, marker='o', linewidth=2, label='Model')
    
    ax1.set_xlabel("Mean Predicted Probability")
    ax1.set_ylabel("Fraction of Positives (Ground Truth)")
    ax1.set_title("Calibration Curve (Reliability)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # --- Plot B: Histogram (Sharpness) ---
    # We use a log scale on Y because binary data is usually dominated by zeros
    ax2.hist(y_prob, range=(0, 1), bins=n_bins, density=True, 
             alpha=0.7, color='purple', edgecolor='black')
    
    ax2.set_xlabel("Predicted Probability")
    ax2.set_ylabel("Density (Log Scale)")
    ax2.set_yscale('log') # Crucial to see the rare high-confidence predictions
    ax2.set_title("Probability Distribution (Sharpness)")
    ax2.grid(True, alpha=0.3, which="both")

    plt.tight_layout()
    plt.show()

plot_calibration_and_sharpness(gt_hits, model_probs, n_bins=20)

In [ ]:
def check_edge_behavior(gt_hits_all, model_probs_all):
    # 1. Calculate the Global Mean per voxel (The "Heatmap")
    # Shape: [num_voxels]
    gt_mean_per_voxel = gt_hits_all.float().mean(dim=0).flatten()
    
    # 2. Identify "Edge Voxels" (The rings)
    # We want voxels that are truly variable (e.g. active 20-80% of the time)
    edge_mask = (gt_mean_per_voxel < 0.2) & (gt_mean_per_voxel > 0.1)
    
    print(f"Total Voxels: {len(edge_mask)}")
    print(f"Edge Voxels (Variable): {edge_mask.sum()} ({edge_mask.float().mean()*100:.1f}%)")
    
    # 3. Get Model Predictions ONLY for these voxels
    # We flatten the batch dimension so we see all predictions for these specific spots
    # Shape: [num_events * num_edge_voxels]
    relevant_model_preds = model_probs_all[:, edge_mask.reshape(model_probs_all.shape[1:])].flatten()
    
    # 4. Plot
    plt.figure(figsize=(10, 6))
    plt.hist(relevant_model_preds.numpy(), bins=50, range=(0, 1), 
             color='red', alpha=0.7, label='Model Predictions (Edge Voxels only)')
    plt.title("What does the model do when it's unsure?")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.legend()
    plt.show()

check_edge_behavior(gt_hits, model_probs)

In [ ]:
def test_conditional_sensitivity(gt_hits_all, model_probs_all, top_k=500):
    """
    Tests if the model assigns higher probabilities when the ground truth is actually active,
    specifically for the "problematic" voxels where the model seems static.
    """
    
    # 1. Compute Variabilities
    gt_std = gt_hits_all.std(dim=0)
    model_std = model_probs_all.std(dim=0)
    
    # 2. Define "High Discrepancy" Voxels
    # We want voxels where GT is noisy (high std) but Model is boring (low std)
    variance_gap = gt_std - model_std
    
    # Get indices of the top K voxels with the worst collapse
    # We flatten to treat the whole detector as a list of voxels
    flat_gap = variance_gap.flatten()
    top_indices = torch.topk(flat_gap, top_k).indices
    
    print(f"Analyzing Top {top_k} 'Collapsed' Voxels...")
    
    # 3. Collect Conditional Probabilities
    # We will aggregate predictions from ALL top_k voxels into two buckets
    probs_given_hit = []
    probs_given_miss = []
    
    # We need to loop or mask carefully. Let's do a masked selection.
    # Convert to Numpy for easier boolean indexing if it fits in memory
    gt_np = gt_hits_all.numpy()
    model_np = model_probs_all.numpy()
    
    # Unravel indices to (row, col) equivalent if needed, but we can index flattened
    # Actually, let's just extract the columns corresponding to these voxels
    # Shape: [num_events, top_k]
    gt_subset = gt_hits_all.view(gt_hits_all.shape[0], -1)[:, top_indices]
    model_subset = model_probs_all.view(model_probs_all.shape[0], -1)[:, top_indices]
    
    # Masking
    # Bucket A: Model Probs where GT was 1
    probs_given_hit = model_subset[gt_subset == 1].numpy()
    
    # Bucket B: Model Probs where GT was 0
    probs_given_miss = model_subset[gt_subset == 0].numpy()
    
    # 4. Visualization
    plt.figure(figsize=(10, 6), dpi=120)
    
    # Plot histograms
    plt.hist(probs_given_miss, bins=50, range=(0, 1), density=True, 
             alpha=0.5, color='red', label=f'GT = 0 (Miss)\nMean Pred: {probs_given_miss.mean():.3f}')
    
    plt.hist(probs_given_hit, bins=50, range=(0, 1), density=True, 
             alpha=0.5, color='blue', label=f'GT = 1 (Hit)\nMean Pred: {probs_given_hit.mean():.3f}')
    
    plt.title(f"Does the Model Know? (Top {top_k} High-Variance-Gap Voxels)")
    plt.xlabel("Model Predicted Probability")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Calculate Separation Metric (Jensen-Shannon or simple difference)
    mean_diff = probs_given_hit.mean() - probs_given_miss.mean()
    plt.annotate(f"Separation (Mean Diff): {mean_diff:.4f}", 
                 xy=(0.5, 0.9), xycoords='axes fraction', ha='center', fontsize=12,
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8))
    
    plt.show()

# Run it
test_conditional_sensitivity(gt_hits, model_probs, top_k=200)

In [ ]:
total_cells = 0
total_hits = 0
# ------------------------------------------------

with torch.no_grad():
        
    for i, (x, x0) in enumerate(self.data_mgr.train_loader):
        batch_hits = (x > 0).float().sum().item() 
        batch_total = x.numel() # Total number of elements (pixels/voxels)
        
        total_hits += batch_hits
        total_cells += batch_total
        # --------------------------------------------------------
    sparsity = 1.0 - (total_hits / total_cells)
    print(f"\n--- DATASET STATISTICS ---")
    print(f"Total Cells: {total_cells}")
    print(f"Total Hits:  {total_hits}")
    print(f"Sparsity (fraction of zeros): {sparsity:.6f}")
    print(f"Recommended Alpha (for hits): {sparsity:.6f}")
    print(f"--------------------------\n")

In [ ]:
with torch.no_grad():
    for i, (x, x0) in enumerate(self.data_mgr.val_loader):
        x = x.to(self.device)
        x0 = x0.to(self.device)
        x_reduce = self._reduce(x, x0)
        # Forward pass
        output = self.model((x_reduce, x0))
        shower = output[3]
        batch_negatives = (shower < 0).float().sum().item()
        print(f"Batch {i} - Negatives in shower output: {batch_negatives}")
